*Note: This notebook was executed on Google Colab (T4 GPU). The modular implementation of the same logic can be found in src/. Results are saved in predictions/translations.json.*

## Load Data

In [ ]:
from datasets import load_dataset

def load_data():
    ds = load_dataset("Helsinki-NLP/europarl", "de-en")
    return ds["train"].shuffle(seed=42).select(range(10000))

def clean_data(ds):
    clean_ds = ds.filter(lambda x:
        len(x["translation"]["en"].split()) >= 3 and
        len(x["translation"]["de"].split()) >= 3 and
        len(x["translation"]["en"].split()) <= 128 and
        len(x["translation"]["de"].split()) <= 128 and
        0.5 <= len(x["translation"]["de"].split()) / len(x["translation"]["en"].split()) <= 2.0
    )
    seen = set()
    def remove_duplicates(example):
        en = example["translation"]["en"]
        if en in seen:
            return False
        seen.add(en)
        return True
    return clean_ds.filter(remove_duplicates)

## Create Preprocessing-Functions

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")

def preprocess_function(examples):
    inputs = [example["en"] for example in examples["translation"]]
    targets = [example["de"] for example in examples["translation"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def tokenize_and_split(ds):
    tokenized = ds.map(preprocess_function, batched=True)
    train_val = tokenized.train_test_split(test_size=0.2, seed=42)
    val_test = train_val["test"].train_test_split(test_size=0.5, seed=42)
    return train_val["train"], val_test["train"], val_test["test"]

## Create Evaluation Functions

In [ ]:
import evaluate as hf_evaluate
import numpy as np
from nltk.translate.meteor_score import meteor_score
from transformers import pipeline

bleu = hf_evaluate.load("sacrebleu")
chrf = hf_evaluate.load("chrf")


def compute_meteor(predictions, references):
    scores = [
        meteor_score([ref.split()], pred.split())
        for pred, ref in zip(predictions, references)
    ]
    return sum(scores) / len(scores)


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels_bleu = [[label.strip()] for label in decoded_labels]
    decoded_labels_flat = [label.strip() for label in decoded_labels]

    return {
        "bleu": bleu.compute(predictions=decoded_preds, references=decoded_labels_bleu)["score"],
        "meteor": compute_meteor(decoded_preds, decoded_labels_flat),
        "chrf": chrf.compute(predictions=decoded_preds, references=decoded_labels_bleu)["score"],
    }


from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def evaluate_nllb(test_sources, test_references):
    model_name = "facebook/nllb-200-distilled-600M"
    nllb_tokenizer = AutoTokenizer.from_pretrained(model_name)
    nllb_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

    predictions = []
    for text in test_sources:
        inputs = nllb_tokenizer(text, return_tensors="pt").to("cuda")
        outputs = nllb_model.generate(**inputs, forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("deu_Latn"), max_length=128)
        predictions.append(nllb_tokenizer.decode(outputs[0], skip_special_tokens=True))

    refs_bleu = [[r] for r in test_references]
    return {
        "bleu": bleu.compute(predictions=predictions, references=refs_bleu)["score"],
        "meteor": compute_meteor(predictions, test_references),
        "chrf": chrf.compute(predictions=predictions, references=refs_bleu)["score"],
    }


def evaluate_marian_pipeline(test_sources, test_references):
    marian_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
    marian_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-de").to("cuda")

    predictions = []
    for text in test_sources:
        inputs = marian_tokenizer(text, return_tensors="pt").to("cuda")
        outputs = marian_model.generate(**inputs, max_length=128)
        predictions.append(marian_tokenizer.decode(outputs[0], skip_special_tokens=True))

    refs_bleu = [[r] for r in test_references]
    return {
        "bleu": bleu.compute(predictions=predictions, references=refs_bleu)["score"],
        "meteor": compute_meteor(predictions, test_references),
        "chrf": chrf.compute(predictions=predictions, references=refs_bleu)["score"],
    }

## Data Cleaning

In [ ]:
def clean_data(ds):
    clean_ds = ds.filter(lambda x:
        len(x["translation"]["en"].split()) >= 3 and
        len(x["translation"]["de"].split()) >= 3 and
        len(x["translation"]["en"].split()) <= 128 and
        len(x["translation"]["de"].split()) <= 128 and
        0.5 <= len(x["translation"]["de"].split()) / len(x["translation"]["en"].split()) <= 2.0
    )
    seen = set()
    def remove_duplicates(example):
        en = example["translation"]["en"]
        if en in seen:
            return False
        seen.add(en)
        return True
    return clean_ds.filter(remove_duplicates)

ds = load_data()
clean_ds = clean_data(ds)
train_ds, val_ds, test_ds = tokenize_and_split(clean_ds)

# Check
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
print(f"Features: {train_ds.column_names}")
print(f"Example labels: {train_ds[0]['labels'][:10]}")

## NLLB Model

In [ ]:
test_sources = [example["en"] for example in test_ds["translation"]]
test_references = [example["de"] for example in test_ds["translation"]]

nllb_results = evaluate_nllb(test_sources, test_references)
print(f"NLLB: {nllb_results}")

## MarianMT Pipeline

In [ ]:
marian_results = evaluate_marian_pipeline(test_sources, test_references)
print(f"Pipeline: {marian_results}")

## Different Hyperparameter Experiments

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

experiments = [
    {"name": "Baseline",      "lr": 2e-5, "epochs": 3, "batch_size": 16, "warmup": 0},
    {"name": "Higher LR",     "lr": 5e-5, "epochs": 3, "batch_size": 16, "warmup": 0},
    {"name": "Lower LR",      "lr": 1e-5, "epochs": 3, "batch_size": 16, "warmup": 0},
    {"name": "More Epochs",   "lr": 2e-5, "epochs": 5, "batch_size": 16, "warmup": 0},
    {"name": "Warmup",        "lr": 2e-5, "epochs": 5, "batch_size": 16, "warmup": 200},
]

results = []

for exp in experiments:
    print(f"\n{'='*50}")
    print(f"Running: {exp['name']}")
    print(f"{'='*50}")

    model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-de")
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./results/{exp['name']}",
        eval_strategy="epoch",
        learning_rate=exp["lr"],
        per_device_train_batch_size=exp["batch_size"],
        per_device_eval_batch_size=exp["batch_size"],
        num_train_epochs=exp["epochs"],
        warmup_steps=exp["warmup"],
        weight_decay=0.01,
        save_total_limit=1,
        predict_with_generate=True,
        generation_max_length=128,
        fp16=True,
        dataloader_pin_memory=False,
        report_to="none",
        logging_strategy="epoch"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_result = trainer.evaluate()
    eval_result["name"] = exp["name"]
    results.append(eval_result)

# Comparison table
import pandas as pd
df = pd.DataFrame(results)[["name", "eval_bleu", "eval_meteor", "eval_chrf", "eval_loss"]]
df.columns = ["Experiment", "BLEU", "METEOR", "chrF", "Loss"]
print(df.to_string(index=False))

# Final Configuration

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-de")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./results/final",
    eval_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    predict_with_generate=True,
    generation_max_length=64,
    fp16=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Final Evaluation on TEST-Set
test_result = trainer.evaluate(test_ds)
print(f"Fine-Tuned Test Results:")
print(f"BLEU:   {test_result['eval_bleu']:.2f}")
print(f"METEOR: {test_result['eval_meteor']:.4f}")
print(f"chrF:   {test_result['eval_chrf']:.2f}")

## Save Predictions

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# NLLB Predictions
nllb_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
nllb_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M").to("cuda")

nllb_predictions = []
for text in test_sources:
    inputs = nllb_tokenizer(text, return_tensors="pt").to("cuda")
    outputs = nllb_model.generate(**inputs, forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("deu_Latn"), max_length=128)
    nllb_predictions.append(nllb_tokenizer.decode(outputs[0], skip_special_tokens=True))
print("NLLB done")

# Pipeline Predictions
marian_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
marian_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-de").to("cuda")

marian_predictions = []
for text in test_sources:
    inputs = marian_tokenizer(text, return_tensors="pt").to("cuda")
    outputs = marian_model.generate(**inputs, max_length=128)
    marian_predictions.append(marian_tokenizer.decode(outputs[0], skip_special_tokens=True))
print("Pipeline done")

# Fine-tuned Predictions
finetuned_predictions = []
for text in test_sources:
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    outputs = trainer.model.generate(**inputs, max_length=128)
    finetuned_predictions.append(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("Fine-tuned done")

In [ ]:
import json
from nltk.translate.meteor_score import meteor_score
import sacrebleu

data = []
for i in range(len(test_sources)):
    entry = {
        "source": test_sources[i],
        "reference": test_references[i],
        "nllb": {
            "translation": nllb_predictions[i],
            "bleu": sacrebleu.sentence_bleu(nllb_predictions[i], [test_references[i]]).score,
            "chrf": sacrebleu.sentence_chrf(nllb_predictions[i], [test_references[i]]).score,
            "meteor": meteor_score([test_references[i].split()], nllb_predictions[i].split()),
        },
        "pipeline": {
            "translation": marian_predictions[i],
            "bleu": sacrebleu.sentence_bleu(marian_predictions[i], [test_references[i]]).score,
            "chrf": sacrebleu.sentence_chrf(marian_predictions[i], [test_references[i]]).score,
            "meteor": meteor_score([test_references[i].split()], marian_predictions[i].split()),
        },
        "finetuned": {
            "translation": finetuned_predictions[i],
            "bleu": sacrebleu.sentence_bleu(finetuned_predictions[i], [test_references[i]]).score,
            "chrf": sacrebleu.sentence_chrf(finetuned_predictions[i], [test_references[i]]).score,
            "meteor": meteor_score([test_references[i].split()], finetuned_predictions[i].split()),
        },
    }
    data.append(entry)

with open("all_translations_with_scores.json", "w") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

## Visualizations

In [ ]:
import matplotlib.pyplot as plt

epochs = [1, 2, 3]
train_loss = [1.358, 1.274, 1.231]
val_loss = [1.280, 1.281, 1.282]

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_loss, marker="o", label="Train Loss")
plt.plot(epochs, val_loss, marker="o", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs. Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

trainer.model.config.attn_implementation = "eager"

model = trainer.model.__class__.from_pretrained(
    "Helsinki-NLP/opus-mt-en-de",
    attn_implementation="eager"
).to("cuda")

model.load_state_dict(trainer.model.state_dict())
model.eval()


source_text = "I concur with his arguments about enlargement."

inputs = tokenizer(source_text, return_tensors="pt").to("cuda")

generated = model.generate(
    **inputs,
    max_new_tokens=64
)

translation = tokenizer.decode(
    generated[0],
    skip_special_tokens=True
)

print("EN:", source_text)
print("DE:", translation)

decoder_input_ids = generated[:, :-1]

with torch.no_grad():

    outputs = model(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        decoder_input_ids=decoder_input_ids,
        output_attentions=True,
        return_dict=True
    )

cross_attentions = outputs.cross_attentions

layer_idx = len(cross_attentions) // 2
head_idx = 3

attn = cross_attentions[layer_idx][0]

print("Raw attention shape:", attn.shape)

attention_matrix = attn[head_idx].cpu().numpy()

print("Final matrix shape:", attention_matrix.shape)

source_tokens = tokenizer.convert_ids_to_tokens(
    inputs.input_ids[0]
)

target_tokens = tokenizer.convert_ids_to_tokens(
    decoder_input_ids[0]
)

source_labels = [
    tok.replace("▁", " ").strip()
    for tok in source_tokens
]

target_labels = [
    tok.replace("▁", " ").strip()
    for tok in target_tokens
]

source_labels = [
    s if s != "" else "▁"
    for s in source_labels
]

target_labels = [
    s if s != "" else "▁"
    for s in target_labels
]

plt.figure(figsize=(12, 8))

im = plt.imshow(
    attention_matrix,
    aspect="auto",
    cmap="Blues"
)

plt.xticks(
    range(len(source_labels)),
    source_labels,
    rotation=45,
    ha="right",
    fontsize=11
)

plt.yticks(
    range(len(target_labels)),
    target_labels,
    fontsize=11
)

plt.xlabel("Source Tokens (EN)")
plt.ylabel("Target Tokens (DE)")

plt.title(
    f"Cross Attention\n"
    f"Layer {layer_idx} | Head {head_idx}"
)

plt.colorbar(im, label="Attention Weight")

plt.tight_layout()
plt.show()
